# CropSense Module 3 — causal pest-risk experiment and audit

**Required architecture:** LSTM. **Status before execution:** no new result claimed.

The original regression notebook used 2009 for early stopping, concatenated training seasons before windowing,
and interpolated missing pest values using later observations. Those results remain historical evidence,
not an untouched forecast benchmark. The deployed artifact is a separate 4×9, three-class model of unresolved provenance.

This experiment uses 2005–2007 for fitting, 2008 for selection, and 2009 once for final comparison.
Windows stay inside a year and contain consecutive weeks. Missing past pest readings use forward fill for at most
four weeks; targets require both pests to have been observed. No future interpolation and no fabricated targets.
Because this changes the evaluation protocol and target availability, new percentages must not be presented as
directly comparable to historical scores on interpolated targets. The small single-region sample is a major limitation.


In [ ]:
PROJECT_MODULE='cropsense-module-3'
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn==1.6.1','lxml'],check=True)


In [ ]:
import os, sys, json, hashlib, platform, subprocess, time
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, log_loss)

SEED = 42
np.random.seed(SEED)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/kaggle/working') / PROJECT_MODULE / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

write_json('environment.json', {'run_id':RUN_ID, 'python':sys.version, 'platform':platform.platform(),
    'packages':{d.metadata['Name']:d.version for d in metadata.distributions()},
    'seed':SEED, 'notebook_status':'executed; results below are generated, not pre-filled'})
results = []

def evaluate(name, split, y_true, probabilities, classes, sample_ids):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    assert probabilities.shape == (len(y_true), len(classes))
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-4)
    pred = probabilities.argmax(axis=1)
    labels = np.arange(len(classes))
    report = classification_report(y_true, pred, labels=labels, target_names=classes,
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=labels)
    present = np.unique(y_true)
    row = {'model':name, 'split':split, 'n':len(y_true),
           'accuracy':float(accuracy_score(y_true,pred)),
           'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
           'macro_f1_all_classes':float(f1_score(y_true,pred,labels=labels,average='macro',zero_division=0)),
           'macro_f1_present_classes':float(f1_score(y_true,pred,labels=present,average='macro',zero_division=0)),
           'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0)),
           'log_loss':float(log_loss(y_true, probabilities, labels=labels))}
    # Wilson interval is descriptive; correlated field tiles/weeks violate independent-sample assumptions.
    n=len(y_true); p=row['accuracy']; z=1.96
    center=(p+z*z/(2*n))/(1+z*z/n)
    radius=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    row['accuracy_wilson95_descriptive']=[float(center-radius),float(center+radius)]
    if classes == ['healthy','moderate_stress','severe_stress']:
        row['healthy_recall']=float(report['healthy']['recall'])
        row['severe_to_healthy_count']=int(cm[2,0])
        row['severe_to_healthy_rate']=float(cm[2,0]/max(cm[2].sum(),1))
    prefix=f'{name}__{split}'
    write_json(prefix+'_metrics.json', {'summary':row,'per_class':report,'class_order':classes,
                                      'confusion_matrix':cm.tolist()})
    pd.DataFrame(report).T.to_csv(OUT/(prefix+'_classification_report.csv'))
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(OUT/(prefix+'_confusion_counts.csv'))
    frame=pd.DataFrame({'sample_id':sample_ids,'true_index':y_true,'pred_index':pred})
    for i,c in enumerate(classes): frame['prob_'+c]=probabilities[:,i]
    frame.to_csv(OUT/(prefix+'_predictions.csv'),index=False)
    fig,axes=plt.subplots(1,2,figsize=(max(12,len(classes)*.65),max(5,len(classes)*.35)))
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    for ax,data,title in zip(axes,[cm,norm],['Counts','Recall by true class']):
        im=ax.imshow(data,cmap='Blues');fig.colorbar(im,ax=ax,fraction=.04)
        ax.set(xticks=labels,yticks=labels,xticklabels=classes,yticklabels=classes,
               xlabel='Predicted',ylabel='True',title=title)
        ax.tick_params(axis='x',rotation=90)
        if len(classes)<=6:
            for r in labels:
                for c in labels: ax.text(c,r,f'{data[r,c]:.2f}' if data is norm else str(data[r,c]),ha='center',va='center')
    fig.suptitle(prefix);fig.tight_layout();fig.savefig(OUT/(prefix+'_confusion.png'),dpi=150);plt.show();plt.close(fig)
    print(pd.DataFrame(report).T.to_string());print(row)
    results.append(row)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    return row

def save_history(name, history):
    pd.DataFrame(history.history).to_csv(OUT/(name+'_history.csv'),index_label='epoch')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axes,['loss','accuracy']):
        for key in [metric,'val_'+metric]:
            if key in history.history: ax.plot(np.arange(1,len(history.history[key])+1),history.history[key],label=key)
        ax.set(xlabel='Epoch',title=metric);ax.legend()
    fig.tight_layout();fig.savefig(OUT/(name+'_learning_curves.png'),dpi=150);plt.show();plt.close(fig)

def finish(model_file, contract, decision):
    write_json('model_contract.json', dict(contract, artifact=Path(model_file).name,
                                         sha256=sha256(model_file),run_id=RUN_ID))
    write_json('decision.json',decision)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    lines=['# '+PROJECT_MODULE+' — executed experiment report', '', 'Run: '+RUN_ID,
           '', '## Results', '', pd.DataFrame(results).to_csv(index=False),
           '## Decision', '', json.dumps(decision,indent=2),
           '', '## Reproduction', 'See environment.json, config.json, split_manifest.csv, model_contract.json, and this executed notebook.',
           '', '## Limitations', contract.get('limitations','')]
    (OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
    import shutil
    archive=shutil.make_archive(str(OUT.parent/(RUN_ID+'_evidence')),'zip',OUT)
    print('Download evidence:',archive)
    print('Candidate export:',model_file,'SHA256:',sha256(model_file))


## 1. Acquire and archive the original records
Use the public ICAR-CRIDA endpoint. Store response bodies and parsed CSVs with hashes, query parameters,
retrieval time, and missing-value counts. A failed request stops the run rather than substituting synthetic data.


In [ ]:
import requests
from io import StringIO
YEAR='Observation Year'; WEEK='Standard Week'
WEATHER=['MaxT(°C)','MinT(°C)','RH1(%)','RH2(%)','RF(mm)','WS(kmph)','SSH(hrs)','EVP(mm)']
PESTS=['Jassid','Aphid']
raw={};provenance={}
for pest in PESTS:
    params={'combo0':'Cotton','combo1':'Akola','combo2':pest,'combo3':'2005','combo4':'2009'}
    response=requests.get('http://icar-crida.res.in:8080/naip/Servlet1.jsp',params=params,timeout=60)
    response.raise_for_status()
    path=OUT/(pest+'_source.html');path.write_bytes(response.content)
    tables=pd.read_html(StringIO(response.text))
    table=next(t for t in tables if {YEAR,WEEK,'Pest Value',*WEATHER}.issubset(t.columns))
    table=table[[YEAR,WEEK,'Pest Value',*WEATHER]].apply(pd.to_numeric,errors='coerce')
    table=table.dropna(subset=[YEAR,WEEK]);table[[YEAR,WEEK]]=table[[YEAR,WEEK]].astype(int)
    assert not table.duplicated([YEAR,WEEK]).any()
    table.to_csv(OUT/(pest+'_raw.csv'),index=False)
    raw[pest]=table.rename(columns={'Pest Value':pest})
    provenance[pest]={'query':params,'sha256_html':sha256(path),'rows':len(table),
                      'missing':table.isna().sum().to_dict(),'retrieved_utc':datetime.now(timezone.utc).isoformat()}
    time.sleep(1)
df=raw['Jassid'].merge(raw['Aphid'][[YEAR,WEEK,'Aphid']],on=[YEAR,WEEK],validate='one_to_one')
df=df.sort_values([YEAR,WEEK]).reset_index(drop=True)
write_json('data_provenance.json',provenance)
print(df.groupby(YEAR)[PESTS].count());print(df.isna().sum())


## 2. Fit target definitions on training years only
The pressure target is the maximum of the two pests after training-only min–max scaling.
Training-target tertiles define low/medium/high. Thresholds and scalers remain fixed for validation and test.
Input weather normalization is embedded in the exported Keras graph, preventing a missing external scaler at deployment.


In [ ]:
TRAIN_YEARS=[2005,2006,2007];VAL_YEAR=2008;TEST_YEAR=2009;WINDOW=4
CLASSES=['low','medium','high'];FEATURES=WEATHER+['pest_pressure']
train_rows=df[YEAR].isin(TRAIN_YEARS)
scale={}
for p in PESTS:
    lo=float(df.loc[train_rows,p].min());hi=float(df.loc[train_rows,p].max())
    assert np.isfinite([lo,hi]).all() and hi>lo
    scale[p]={'min':lo,'max':hi}
    df[p+'_norm']=((df[p]-lo)/(hi-lo)).clip(0,1)
df['target_pressure']=df[[p+'_norm' for p in PESTS]].max(axis=1,skipna=False)
for p in PESTS:
    df[p+'_past']=df.groupby(YEAR)[p+'_norm'].ffill(limit=WINDOW)
df['pest_pressure']=df[[p+'_past' for p in PESTS]].max(axis=1,skipna=False)
thresholds=np.quantile(df.loc[train_rows,'target_pressure'].dropna(),[1/3,2/3])
assert thresholds[0]<thresholds[1], 'Insufficient target diversity.'

def windows(frame):
    x=[];y=[];ids=[];pressures=[]
    for year,season in frame.groupby(YEAR,sort=True):
        season=season.sort_values(WEEK)
        for i in range(WINDOW,len(season)):
            segment=season.iloc[i-WINDOW:i+1]
            if not np.all(np.diff(segment[WEEK])==1): continue
            past=segment.iloc[:-1][FEATURES].to_numpy(dtype='float32')
            target=float(segment.iloc[-1]['target_pressure'])
            if not np.isfinite(past).all() or not np.isfinite(target): continue
            x.append(past);y.append(int(np.digitize(target,thresholds)));pressures.append(target)
            ids.append(f'{int(year)}-W{int(segment.iloc[-1][WEEK]):02d}')
    return np.asarray(x,dtype='float32'),np.asarray(y),ids,np.asarray(pressures)

parts={}
for split,mask in [('train',df[YEAR].isin(TRAIN_YEARS)),('validation',df[YEAR]==VAL_YEAR),('test',df[YEAR]==TEST_YEAR)]:
    parts[split]=windows(df[mask])
    x,y,ids,_=parts[split]
    assert len(y)>0 and x.shape[1:]==(4,9), f'No valid {split} windows.'
    print(split,x.shape,'class counts',np.bincount(y,minlength=3))
assert set(parts['train'][1])=={0,1,2},'All classes must be represented in training.'
pd.DataFrame([{'sample_id':sid,'split':split,'class_index':int(y)} for split,(_,ys,ids,_) in parts.items()
              for sid,y in zip(ids,ys)]).to_csv(OUT/'split_manifest.csv',index=False)
df.to_csv(OUT/'processed_weekly_data.csv',index=False)
config={'train_years':TRAIN_YEARS,'validation_year':VAL_YEAR,'test_year':TEST_YEAR,'features':FEATURES,
        'window':WINDOW,'thresholds':thresholds.tolist(),'pest_scaling':scale,
        'missing_input_rule':'within-year forward-fill at most 4 weeks','target_rule':'both pest readings observed',
        'candidates':[{'units':8,'dropout':0.1},{'units':16,'dropout':0.3}],
        'selection':'validation macro F1; test never used for early stopping'}
write_json('config.json',config)


## 3. Baselines and bounded LSTM experiment
Compare training-majority, last-observation persistence, and a standardized logistic regression with the required LSTM.
Only validation performance selects between the two LSTM configurations. A good comparison model does not silently replace the required architecture.


In [ ]:
import tensorflow as tf
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
tf.keras.utils.set_random_seed(SEED)
Xtr,ytr,_,_=parts['train'];Xv,yv,vids,_=parts['validation']
majority=int(np.bincount(ytr,minlength=3).argmax())
linear=make_pipeline(StandardScaler(),LogisticRegression(C=0.1,class_weight='balanced',max_iter=2000,random_state=SEED))
linear.fit(Xtr.reshape(len(Xtr),-1),ytr)
weights=dict(enumerate(compute_class_weight('balanced',classes=np.arange(3),y=ytr)))
candidate_models={};validation_rows=[]
for spec in config['candidates']:
    tf.keras.utils.set_random_seed(SEED)
    normalizer=tf.keras.layers.Normalization(axis=-1)
    normalizer.adapt(Xtr.reshape(-1,len(FEATURES)))
    model=tf.keras.Sequential([tf.keras.layers.Input((WINDOW,len(FEATURES))),normalizer,
        tf.keras.layers.LSTM(spec['units'],dropout=spec['dropout']),
        tf.keras.layers.Dense(3,activation='softmax')])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001,clipnorm=1.0),
                  loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    name=f"lstm_u{spec['units']}_d{spec['dropout']}"
    start=time.perf_counter()
    history=model.fit(Xtr,ytr,validation_data=(Xv,yv),epochs=60,batch_size=8,shuffle=False,
        class_weight=weights,callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=8,restore_best_weights=True)],verbose=2)
    save_history(name,history)
    row=evaluate(name,'validation',yv,model.predict(Xv,verbose=0),CLASSES,vids)
    row['training_seconds']=time.perf_counter()-start
    row['best_epoch']=int(np.argmin(history.history['val_loss'])+1)
    validation_rows.append(row);candidate_models[name]=model
selected=max(validation_rows,key=lambda r:(r['macro_f1_all_classes'],r['accuracy']))['model']
write_json('selection_before_test.json',{'selected':selected,'validation_candidates':validation_rows})
print('Selected before test evaluation:',selected)


## 4. Final evaluation, export, and reload check
Report all three baselines and the selected LSTM on the same observed-target test weeks.
The export is a candidate, not an automatic replacement. Compare against persistence before claiming forecasting value.


In [ ]:
Xt,yt,tids,pressure=parts['test']
test_rows={}
test_rows['majority']=evaluate('majority','test',yt,np.eye(3)[np.full(len(yt),majority)],CLASSES,tids)
persistent=np.digitize(Xt[:,-1,-1],thresholds)
test_rows['persistence']=evaluate('persistence','test',yt,np.eye(3)[persistent],CLASSES,tids)
test_rows['logistic']=evaluate('logistic','test',yt,linear.predict_proba(Xt.reshape(len(Xt),-1)),CLASSES,tids)
model=candidate_models[selected]
probs=model.predict(Xt,verbose=0)
test_rows['lstm']=evaluate(selected,'test',yt,probs,CLASSES,tids)
export=OUT/'module3_lstm_candidate.keras';model.save(export)
reloaded=tf.keras.models.load_model(export,compile=False)
np.testing.assert_allclose(probs,reloaded.predict(Xt,verbose=0),atol=1e-6)
samples={'feature_names':FEATURES,'window_size':WINDOW,'samples':[
    {'label':sid,'sequence':x.tolist(),'actual_class':CLASSES[int(y)]} for sid,x,y in zip(tids,Xt,yt)]}
write_json('sample_sequences_candidate.json',samples)
finish(export,{'input_shape':[None,4,9],'output_shape':[None,3],'class_order':CLASSES,'feature_order':FEATURES,
    'input_preprocessing':'raw weather plus training-scaled pest_pressure; weather normalization embedded in graph',
    'pest_scaling':scale,'thresholds':thresholds.tolist(),
    'limitations':'Single region; small test set; historically inspected 2009 season; causal observed-target protocol differs from legacy evaluation.'},
    {'selected':selected,'required_architecture':'LSTM','reload_predictions_match':True,
     'beats_persistence_accuracy':test_rows['lstm']['accuracy']>test_rows['persistence']['accuracy'],
     'beats_persistence_macro_f1':test_rows['lstm']['macro_f1_all_classes']>test_rows['persistence']['macro_f1_all_classes'],
     'deployment':'candidate only; preserve fallback until Windows/Mac integration and provenance review pass'})
